In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from xgboost import XGBClassifier
import re, html
from tqdm.auto import tqdm
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
tqdm.pandas()
nltk.download("punkt", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download('averaged_perceptron_tagger_eng')

%matplotlib inline

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [2]:
df = pd.read_csv(
    "/kaggle/input/ttic-31020-2025a-hw-2-spam-detection/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"],
    quoting=3,
    on_bad_lines="skip"
)

In [3]:
df.head()

,label,message
0,ham,"Aight will do, thanks again for comin out"
1,ham,No..but heard abt tat..
2,spam,Please call our customer service representativ...
3,ham,Yes..he is really great..bhaji told kallis bes...
4,ham,&lt;#&gt; am I think? Should say on syllabus


In [4]:
df.shape

(4574, 2)

In [5]:
df.isnull().sum()

label      0
message    0
dtype: int64

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4574 entries, 0 to 4573
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   label    4574 non-null   object
 1   message  4574 non-null   object
dtypes: object(2)
memory usage: 71.6+ KB


In [7]:
df["label"].value_counts()

label
ham     3979
spam     595
Name: count, dtype: int64

# Map labels: ham -> -1, spam -> 1

In [8]:
df['label'] = df['label'].map({'ham': -1, 'spam': 1})

In [9]:
STOP = set(stopwords.words("english"))

def clean(text):
    if not isinstance(text, str): 
        return ""
    text = html.unescape(text)
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^\w\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [t for t in word_tokenize(text) if t not in STOP and len(t) > 1]
    return " ".join(tokens)

In [10]:
# df["message"] = df["message"].progress_apply(clean)

In [11]:
df.head()

,label,message
0,-1,"Aight will do, thanks again for comin out"
1,-1,No..but heard abt tat..
2,1,Please call our customer service representativ...
3,-1,Yes..he is really great..bhaji told kallis bes...
4,-1,&lt;#&gt; am I think? Should say on syllabus


In [12]:
# Features and labels
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [13]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)

In [14]:
model = CatBoostClassifier(iterations=1000,depth=6,learning_rate=0.005,loss_function="Logloss",verbose=100,random_seed=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

0:	learn: 0.6880530	total: 110ms	remaining: 1m 50s
100:	learn: 0.3433447	total: 3.6s	remaining: 32s
200:	learn: 0.2250094	total: 7.12s	remaining: 28.3s
300:	learn: 0.1733570	total: 10.7s	remaining: 24.9s
400:	learn: 0.1452185	total: 14.2s	remaining: 21.3s
500:	learn: 0.1265947	total: 17.9s	remaining: 17.8s
600:	learn: 0.1145408	total: 21.4s	remaining: 14.2s
700:	learn: 0.1059892	total: 25s	remaining: 10.7s
800:	learn: 0.0994663	total: 28.7s	remaining: 7.12s
900:	learn: 0.0940618	total: 32.3s	remaining: 3.55s
999:	learn: 0.0892411	total: 36s	remaining: 0us


In [15]:
# Accuracy
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

# Precision
prec = precision_score(y_test, y_pred, average='binary')  
print("Precision:", prec)

# Recall
rec = recall_score(y_test, y_pred, average='binary')
print("Recall:", rec)

# F1-score
f1 = f1_score(y_test, y_pred, average='binary')
print("F1-score:", f1)

Accuracy: 0.9781420765027322
Precision: 0.9819819819819819
Recall: 0.8582677165354331
F1-score: 0.9159663865546219


In [16]:
test_df = pd.read_csv(
    "/kaggle/input/ttic-31020-2025a-hw-2-spam-detection/SMSSpamCollection_test_text",
    sep="\t",
    header=None,
    names=["label", "message"],   # <-- two columns
    quoting=3,
    on_bad_lines="skip"
)
test_df = test_df.reset_index().rename(columns={"index": "id"})

In [17]:
test_df.head()

,id,label,message
0,0,0,"Go until jurong point, crazy.. Available only ..."
1,1,0,Ok lar... Joking wif u oni...
2,2,0,Free entry in 2 a wkly comp to win FA Cup fina...
3,3,0,U dun say so early hor... U c already then say...
4,4,0,"Nah I don't think he goes to usf, he lives aro..."


In [18]:
test_df["id"] = range(len(test_df))

# Expand to 2262 rows
total_required = 2262
if len(test_df) < total_required:
    extra_rows = pd.DataFrame({"id": range(len(test_df), total_required),"label": 0,"message": ""})
    test_df = pd.concat([test_df, extra_rows], ignore_index=True)

In [19]:
test_df.shape

(2262, 3)

In [20]:
Id=test_df.id

In [21]:
test_df.drop(columns=["id","label"],axis=1,inplace=True)

In [22]:
X_test_pred=tfidf.transform(test_df["message"])
pred=model.predict(X_test_pred)
submission=pd.DataFrame({"ID":Id,"LABEL":pred})
submission.to_csv("submission.csv",index=False)
submission.head()

,ID,LABEL
0,0,-1
1,1,-1
2,2,1
3,3,-1
4,4,-1


# Model 2

In [23]:
df["label"] = df["label"].replace(-1, 0)

In [24]:
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)

In [26]:
xgb = XGBClassifier(n_estimators=1000,learning_rate=0.005,max_depth=6,random_state=42,booster='gbtree',tree_method='auto')
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

In [27]:
# Accuracy
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", acc)

# Precision
prec = precision_score(y_test, y_pred, average='binary')  
print("Precision:", prec)

# Recall
rec = recall_score(y_test, y_pred, average='binary')
print("Recall:", rec)

# F1-score
f1 = f1_score(y_test, y_pred, average='binary')
print("F1-score:", f1)

Accuracy: 0.9737704918032787
Precision: 0.9478260869565217
Recall: 0.8582677165354331
F1-score: 0.9008264462809917


In [28]:
X_test_pred=tfidf.transform(test_df["message"])
pred=xgb.predict(X_test_pred)
submission=pd.DataFrame({"ID":Id,"LABEL":pred})
submission["LABEL"] = submission["LABEL"].replace(0,-1)
submission.to_csv("new_submission.csv",index=False)
submission.head()

,ID,LABEL
0,0,-1
1,1,-1
2,2,1
3,3,-1
4,4,-1


# Model 3

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import Counter
from tqdm import tqdm
from tabulate import tabulate
import nltk
from torchsummary import summary
from nltk.tokenize import word_tokenize
nltk.download('punkt')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [30]:
df = pd.read_csv(
    "/kaggle/input/ttic-31020-2025a-hw-2-spam-detection/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"],
    quoting=3,
    on_bad_lines="skip"
)

In [31]:
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

In [32]:
df.head()

,label,message
0,0,"Aight will do, thanks again for comin out"
1,0,No..but heard abt tat..
2,1,Please call our customer service representativ...
3,0,Yes..he is really great..bhaji told kallis bes...
4,0,&lt;#&gt; am I think? Should say on syllabus


In [33]:
def tokenize(text):
    return word_tokenize(text.lower())

# Build vocabulary

In [34]:
def build_vocab(texts):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    vocab = {word: idx + 2 for idx, (word, _) in enumerate(counter.most_common())}
    vocab['<PAD>'] = 0  # Padding token
    vocab['<UNK>'] = 1  # Unknown token
    return vocab

vocab = build_vocab(df['message'])

# Custom Dataset

In [35]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=50):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = tokenize(self.texts[idx])
        # Convert words to indices
        indices = [self.vocab.get(word, self.vocab['<UNK>']) for word in text]
        # Pad or truncate to max_len
        if len(indices) < self.max_len:
            indices += [self.vocab['<PAD>']] * (self.max_len - len(indices))
        else:
            indices = indices[:self.max_len]
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

In [36]:
train_texts, val_texts, train_labels, val_labels = train_test_split(df['message'].values, df['label'].values, test_size=0.2, random_state=42)

In [37]:
batch_size=32
train_dataset = TextDataset(train_texts, train_labels, vocab)
val_dataset = TextDataset(val_texts, val_labels, vocab)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

In [38]:
X_batch, y_batch = next(iter(train_loader))
print(X_batch.shape, y_batch.shape)

torch.Size([32, 50]) torch.Size([32])


# Define the LSTM Model



In [39]:
import torch
import torch.nn as nn
from torchinfo import summary

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, text):
        embedded = self.embedding(text.long())
        _, (hidden, _) = self.lstm(embedded)
        hidden = hidden.squeeze(0)
        return self.fc(hidden)

vocab_size = 10000
embed_dim = 100
hidden_dim = 128
output_dim = 2

model = LSTMClassifier(vocab_size, embed_dim, hidden_dim, output_dim)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

summary(model,input_size=(32, 50),device=device,verbose=1,col_names=["input_size", "output_size", "num_params"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
LSTMClassifier                           [32, 50]                  [32, 2]                   --
├─Embedding: 1-1                         [32, 50]                  [32, 50, 100]             1,000,000
├─LSTM: 1-2                              [32, 50, 100]             [32, 50, 128]             117,760
├─Linear: 1-3                            [32, 128]                 [32, 2]                   258
Total params: 1,118,018
Trainable params: 1,118,018
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 220.42
Input size (MB): 0.01
Forward/backward pass size (MB): 2.92
Params size (MB): 4.47
Estimated Total Size (MB): 7.40


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
LSTMClassifier                           [32, 50]                  [32, 2]                   --
├─Embedding: 1-1                         [32, 50]                  [32, 50, 100]             1,000,000
├─LSTM: 1-2                              [32, 50, 100]             [32, 50, 128]             117,760
├─Linear: 1-3                            [32, 128]                 [32, 2]                   258
Total params: 1,118,018
Trainable params: 1,118,018
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 220.42
Input size (MB): 0.01
Forward/backward pass size (MB): 2.92
Params size (MB): 4.47
Estimated Total Size (MB): 7.40

# Loss and optimizer

In [40]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)



# Training function

In [41]:
def train_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for texts, labels in tqdm(data_loader, desc="Training"):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    return total_loss / len(data_loader), correct / total


# Validation function

In [42]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for texts, labels in tqdm(data_loader, desc="Validating"):
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return total_loss / len(data_loader), correct / total

# Display results with tabulate

In [43]:
patience = 5
best_val_loss = float('inf')
counter = 0

num_epochs = 100
results = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    results.append([epoch + 1, train_loss, train_acc, val_loss, val_acc])
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pth")  # save best model
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break


headers = ["Epoch", "Train Loss", "Train Accuracy", "Val Loss", "Val Accuracy"]
print(tabulate(results, headers=headers, tablefmt="grid"))

Validating: 100%|██████████| 29/29 [00:00<00:00, 71.44it/s]

+---------+--------------+------------------+------------+----------------+
|   Epoch |   Train Loss |   Train Accuracy |   Val Loss |   Val Accuracy |
+=========+==============+==================+============+================+
|       1 |    0.682589  |         0.863351 |   0.6762   |       0.853552 |
+---------+--------------+------------------+------------+----------------+
|       2 |    0.668687  |         0.867723 |   0.662421 |       0.855738 |
+---------+--------------+------------------+------------+----------------+
|       3 |    0.653513  |         0.869637 |   0.647008 |       0.861202 |
+---------+--------------+------------------+------------+----------------+
|       4 |    0.63561   |         0.87155  |   0.627778 |       0.861202 |
+---------+--------------+------------------+------------+----------------+
|       5 |    0.611879  |         0.872096 |   0.59926  |       0.861202 |
+---------+--------------+------------------+------------+----------------+
|       6 | 

In [44]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import torch

def predict_and_evaluate(model, data_loader, device):
    model.eval()
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for texts, labels in data_loader:
            texts, labels = texts.to(device), labels.to(device)
            outputs = model(texts)
            _, predicted = torch.max(outputs, 1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())

    # Metrics
    acc = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted")
    report = classification_report(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)

    print("📊 Validation Metrics")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print("\nClassification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)

    return acc, precision, recall, f1, cm


acc, prec, rec, f1, cm = predict_and_evaluate(model, val_loader, device)

📊 Validation Metrics
Accuracy:  0.9694
Precision: 0.9688
Recall:    0.9694
F1-score:  0.9689

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       788
           1       0.92      0.85      0.89       127

    accuracy                           0.97       915
   macro avg       0.95      0.92      0.93       915
weighted avg       0.97      0.97      0.97       915

Confusion Matrix:
[[779   9]
 [ 19 108]]


# Predict ON Test Data

In [45]:
test_df = pd.read_csv(
    "/kaggle/input/ttic-31020-2025a-hw-2-spam-detection/SMSSpamCollection_test_text",
    sep="\t",
    header=None,
    names=["label", "message"],   # <-- two columns
    quoting=3,
    on_bad_lines="skip"
)
test_df = test_df.reset_index().rename(columns={"index": "id"})

In [46]:
test_df["id"] = range(len(test_df))

# Expand to 2262 rows
total_required = 2262
if len(test_df) < total_required:
    extra_rows = pd.DataFrame({"id": range(len(test_df), total_required),"label": 0,"message": ""})
    test_df = pd.concat([test_df, extra_rows], ignore_index=True)

In [47]:
test_df.drop(columns=["label","id"],axis=1,inplace=True)

In [48]:
test_df.head()

,message
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
2,Free entry in 2 a wkly comp to win FA Cup fina...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."


In [49]:
class TestDataset(Dataset):
    def __init__(self, texts, vocab, max_len=50):
        self.texts = texts
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = tokenize(self.texts[idx])
        # Convert words to indices
        indices = [self.vocab.get(word, self.vocab['<UNK>']) for word in text]
        # Pad or truncate
        if len(indices) < self.max_len:
            indices += [self.vocab['<PAD>']] * (self.max_len - len(indices))
        else:
            indices = indices[:self.max_len]
        return torch.tensor(indices, dtype=torch.long)

# Create test dataset & dataloader
test_texts = test_df['message'].values
test_dataset = TestDataset(test_texts, vocab)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [50]:
model.eval()
all_preds = []

with torch.no_grad():
    for texts in test_loader:
        texts = texts.to(device)
        outputs = model(texts)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())

# Create submission
submission = pd.DataFrame({"ID": range(len(all_preds)), "LABEL": all_preds})
submission["LABEL"] = submission["LABEL"].replace(0, -1)
submission.to_csv("torch_submission.csv", index=False)
submission.head()

,ID,LABEL
0,0,-1
1,1,-1
2,2,1
3,3,-1
4,4,-1
